<a id='toc3__'></a>

## List and filter available tests

Before we run a test, let's find a suitable test for this demonstration. Let's assume you want to generate the *pearson correlation matrix* for a dataset. A Pearson correlation matrix is a table that shows the [Pearson correlation coefficients](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient) between several variables. 

In the [Explore tests](../explore_tests/explore_tests.ipynb) notebook, we learned how to pass a `filter` to the `list_tests` function. We'll do the same here to find the test ID for the pearson correlation matrix:

In [ ]:
vm.tests.list_tests(filter="PearsonCorrelationMatrix")

From the output, you can see that the test ID for the pearson correlation matrix is `validmind.data_validation.PearsonCorrelationMatrix`. The `describe_test` function gives you more information about the test, including its **Required Inputs**:

In [ ]:
test_id = "validmind.data_validation.PearsonCorrelationMatrix"
vm.tests.describe_test(test_id)

Since this test requires a dataset, it should throw an error if you were to run it without passing a `dataset` input:

In [ ]:
try:
    vm.tests.run_test(test_id)
except Exception as e:
    print(e)

<a id='toc4__'></a>

## Create a sample dataset

Now, let's build a sample dataset so you can generate its pearson correlation matrix. The sklearn `make_classification` function can generate a random dataset for testing:

In [ ]:
import pandas as pd
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=10000,
    n_features=10,
    weights=[0.1],
    random_state=42,
)
X.shape
y.shape

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y
df.head()

<a id='toc4_1__'></a>

### Initialize a ValidMind dataset

ValidMind dataset objects provide a wrapper to any type of dataset (NumPy, Pandas, Polars, etc.) so that tests can run transparently regardless of the underlying library. A VM dataset object can be created using the [`init_dataset`](https://docs.validmind.ai/validmind/validmind.html#init_dataset) function from the ValidMind (`vm`) module.

This function takes a number of arguments:

- `dataset`: The raw dataset that you want to provide as input to tests.
- `input_id`: A unique identifier that allows tracking what inputs are used when running each individual test.
- `target_column`: A required argument if tests require access to true values. This is the name of the target column in the dataset.

Below you can see how to initialize a VM dataset for the sample `df` you created previously:

In [ ]:
vm_dataset = vm.init_dataset(
    df,
    input_id="my_demo_dataset",
    target_column="target",
)

<a id='toc4_2__'></a>

### Run tests with the dataset

You can now call `run_test` with the new `vm_dataset` object as input:

In [ ]:
result = vm.tests.run_test(
    test_id,
    inputs={"dataset": vm_dataset},
)

This dataset can also be used for any other test that requires a dataset input. 

- Let's try to find a *class imbalance* to understand the distribution of the target column in the dataset. 
- Class imbalance is a common problem in machine learning, particularly in classification tasks, where the number of instances (or data points) in each class isn't evenly distributed across the available categories.

We'll use `list_tests` again to showcase how to filter tests for tabular data:

In [ ]:
sorted(vm.tests.list_tags())

In [ ]:
vm.tests.list_tests(tags=["binary_classification", "tabular_data"])

<a id='toc4_2_1__'></a>

#### Run a test that accepts parameters

The test ID for the class imbalance test is `validmind.data_validation.ClassImbalance`. If you describe this test you will find that it also accepts some parameters:

In [ ]:
vm.tests.describe_test("validmind.data_validation.ClassImbalance")

The `min_percent_threshold` will allow you configure the threshold for an acceptable class imbalance. 

- Let's run the test without any parameters to see its output using a default value for the threshold. 
- We also call the `log` method on the result to send the results of the tests to the ValidMind Platform.

In [ ]:
result = vm.tests.run_test(
    "validmind.data_validation.ClassImbalance",
    inputs={"dataset": vm_dataset},
)

result.log()

This test passes the pass-fail criteria with the default threshold of 10%. 

- Let's try to run the test with a threshold of 20% to see if it fails. Notice the use of the 'custom_threshold' `result_id` in the test ID. 
- This allows you to submit individual results for the same test to the ValidMind Platform, as we'll see in the next section.

In [ ]:
result = vm.tests.run_test(
    "validmind.data_validation.ClassImbalance:custom_threshold",
    inputs={"dataset": vm_dataset},
    params={"min_percent_threshold": 20},
)

result.log()

<a id='toc5__'></a>

## Add test results to documentation

The previous result shows that the test didn't pass the threshold of 20% for class imbalance. With these results logged, you can now add them to your model documentation:

1. In the ValidMind Platform, click **Documentation** under Documents for the model you registered earlier. ([Need more help?](https://docs.validmind.ai/guide/model-documentation/working-with-model-documentation.html)

2. Expand the **2.1. Data Description** section.

3. Hover between any existing content blocks to reveal the **+** button.

4. Click on the **+** button and choose **Test-Driven Block**. This will open a dialog where you can select:
    1. Type: `Threshold Test` 
    2. Threshold Test: `Class Imbalance Custom Threshold Test` 
    - You can preview the result and then click **Insert Block** in the bottom-right corner to add it to the documentation.

<a id='toc6__'></a>

## Next steps

While you can look at the results of this test suite right in the notebook where you ran the code, there is a better way — expand the rest of your documentation in the ValidMind Platform and take a look around. 

What you see is the full draft of your model documentation in a more easily consumable version. From here, you can make qualitative edits to model documentation, view guidelines, collaborate with validators, and submit your model documentation for approval when it's ready.

<a id='toc6_1__'></a>

### Discover more learning resources

In an upcoming companion notebook, you'll learn how to run tests that require a dataset and model as inputs. This will allow you to generate documentation for model evaluation tests such as ROC-AUC, F1 score, etc.

We also offer many other interactive notebooks to help you document models:

- [Run tests & test suites](https://docs.validmind.ai/developer/how-to/testing-overview.html)
- [Use ValidMind Library features](https://docs.validmind.ai/developer/how-to/feature-overview.html)
- [Code samples by use case](https://docs.validmind.ai/guide/samples-jupyter-notebooks.html)

Or, visit our [documentation](https://docs.validmind.ai/) to learn more about ValidMind.

<a id='toc7__'></a>

## Upgrade ValidMind

<div class="alert alert-block alert-info" style="background-color: #B5B5B510; color: black; border: 1px solid #083E44; border-left-width: 5px; box-shadow: 2px 2px 4px rgba(0, 0, 0, 0.2);border-radius: 5px;">After installing ValidMind, you’ll want to periodically make sure you are on the latest version to access any new features and other enhancements.</div>

Retrieve the information for the currently installed version of ValidMind:

In [ ]:
%pip show validmind

If the version returned is lower than the version indicated in our [production open-source code](https://github.com/validmind/validmind-library/blob/prod/validmind/__version__.py), restart your notebook and run:

```bash
%pip install --upgrade validmind
```

You may need to restart your kernel after running the upgrade package for changes to be applied.

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>